# Finance Document RAG
Updated original notebook with semantic chunking and hybrid retrieval.

In [ ]:
# CELL 1: ENVIRONMENT SETUP
# install all required packages
# upgrade pip first to avoid package issues
# install llama-cpp for local LLM inference
# install PDF tools, OCR tools, embeddings, vector search, and UI libraries
# check if CUDA / GPU is available
# print system info so runtime is easier to debug

!pip -q install --upgrade pip
!pip -q install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!pip -q install gradio gradio_pdf pymupdf PyPDF2 pillow pytesseract
!pip -q install sentence-transformers faiss-cpu nltk

import torch, llama_cpp
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("llama.cpp info:", llama_cpp.llama_print_system_info().decode("utf-8"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.5 MB/s eta 0:00:00


ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB


CUDA available: True
GPU: Tesla T4
llama.cpp info: CUDA : ARCHS = 700,750,800,860,890,900 | FORCE_MMQ = 1 | USE_GRAPHS = 1 | PEER_MAX_BATCH_SIZE = 128 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 


In [ ]:
# CELL 2: DOWNLOAD MISTRAL-7B GGUF
# create folder to store the model
# define path for quantized Mistral GGUF model
# download model only if it is not already cached
# list model folder to confirm download worked

from pathlib import Path

MODEL_DIR = Path("/content/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
GGUF_PATH = MODEL_DIR / "mistral-7b-instruct-v0.2.Q4_K_M.gguf"

if not GGUF_PATH.exists():
    print("Downloading Mistral-7B-Instruct Q4_K_M (~4.1GB)...")
    !wget -q --show-progress -O {GGUF_PATH} \
      "https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf"
else:
    print("Model already cached.")
!ls -lh {MODEL_DIR}

/content/models/mis 100%[===================>]   4.07G  98.2MB/s    in 24s     
total 4.1G
-rw-r--r-- 1 root root 4.1G Apr  8 08:20 mistral-7b-instruct-v0.2.Q4_K_M.gguf


In [ ]:
# CELL 3: IMPORTS, MODEL LOADING & INFERENCE WRAPPER
# import all libraries used across the pipeline
# download NLTK sentence tokenizer files
# store all main settings in CFG
# load sentence embedding model on GPU
# load local Mistral model with llama.cpp
# create reusable generate() helper for LLM calls
import os, io, json, re, math
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Any
from datetime import datetime
import numpy as np
import fitz
from PIL import Image
import pytesseract
import faiss
import nltk
from sentence_transformers import SentenceTransformer
from llama_cpp import Llama
import gradio as gr
from gradio_pdf import PDF

import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.tokenize import sent_tokenize

CFG = {
    "gguf_path": str(GGUF_PATH),
    "ctx_window": 4096,
    "max_gen_tokens": 600,
    "temperature": 0.1,
    "stop": ["</s>"],
    "gpu_layers": 999,
    "embed_model": "all-MiniLM-L6-v2",
    "semantic_similarity_threshold": 0.60,
    "chunk_min_sentences": 3,
    "chunk_max_sentences": 8,
    "chunk_min_chars": 250,
    "chunk_max_chars": 1600,
    "retrieval_k": 12,
    "hybrid_alpha": 0.55,
    "hybrid_candidate_pool": 10,
}

encoder = SentenceTransformer(CFG["embed_model"], device="cuda")
EMB_DIM = encoder.get_sentence_embedding_dimension()
print(f"Encoder: {CFG['embed_model']} ({EMB_DIM}-dim, GPU)")

assert os.path.exists(CFG["gguf_path"]), f"GGUF not found: {CFG['gguf_path']}"
llm = Llama(
    model_path=CFG["gguf_path"],
    n_ctx=CFG["ctx_window"],
    n_gpu_layers=CFG["gpu_layers"],
    verbose=False,
)
print(f"Mistral-7B loaded (ctx={CFG['ctx_window']})")

def generate(prompt: str, max_tokens: int = None, temp: float = None) -> str:
    resp = llm(
        prompt=prompt,
        max_tokens=max_tokens or CFG["max_gen_tokens"],
        temperature=temp or CFG["temperature"],
        stop=CFG["stop"],
        echo=False,
    )
    return (resp.get("choices", [{}])[0].get("text") or "").strip()

print("Inference ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoder: all-MiniLM-L6-v2 (384-dim, GPU)


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Mistral-7B loaded (ctx=4096)
Inference ready.


In [ ]:
# CELL 4: EMBEDDING FINE-TUNING FOR MORTGAGE/FINANCE DOMAIN
# define finance question-passage training examples
# convert examples into sentence-transformer training format
# fine-tune embedding model on mortgage-style Q/A pairs
# save tuned embedding model
# compare base vs tuned retrieval behavior on sample queries
# replace original encoder with tuned encoder

from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

MORTGAGE_PAIRS = [
    ("What interest rate am I locked at?", "The fixed interest rate for this mortgage is 6.375% per annum for the full 30-year term."),
    ("Tell me the APR on this loan", "APR: 6.582%. This annual percentage rate includes the base interest rate plus lender fees."),
    ("Is my rate adjustable or fixed?", "Loan Type: Conventional Fixed-Rate mortgage. The 6.375% interest rate remains constant."),
    ("Break down my monthly payment for me", "Monthly Principal and Interest: $1,497.23. Estimated Escrow: $387.50 covering taxes and insurance."),
    ("How much goes to escrow each month?", "Estimated Escrow: $387.50 per month. Property Taxes: $262.50/month. Homeowners Insurance: $125.00/month."),
    ("What is the principal and interest portion?", "Monthly Principal and Interest: $1,497.23. This amount remains fixed for the full loan term."),
    ("How much are my closing costs?", "Total Closing Costs: $8,903.18. Origination Charges: $2,400.00. Services: $3,253.18."),
    ("What cash do I need at closing?", "Cash to Close: $68,403.18. Down Payment: $60,000.00 (20%). Closing Costs: $8,403.18."),
    ("What did the lender charge for origination?", "Origination Charges: $2,400.00. This is a 1% loan origination fee calculated on the loan amount."),
    ("List the government recording fees", "Government Recording and Transfer Charges: Recording Fees $250.00, Transfer Tax $500.00."),
    ("How much am I borrowing?", "Loan Amount: $240,000.00 on a property valued at $300,000.00 with 20% down payment."),
    ("When does this loan mature?", "Loan Term: 360 months (30 years). First Payment: March 1, 2025. Final Payment: February 1, 2055."),
    ("What will I pay in total over the full term?", "Total of Payments: $539,003.06. Total Interest Percentage (TIP): 124.01%."),
    ("Can I pay this off early without a fee?", "Prepayment Penalty: NO. The borrower may prepay all or any portion of the principal at any time."),
    ("What is the late fee?", "Late Charge: 5% of the overdue principal and interest payment if not received within 15 days."),
    ("Whose name is on the mortgage?", "Borrower: James A. Richardson and Maria L. Richardson. Property: 742 Evergreen Terrace."),
    ("Which company is the lender?", "Lender: National Home Lending Corp, 500 Finance Plaza, Chicago, IL 60601."),
    ("Who handled the title and closing?", "Settlement Agent: First American Title Insurance Company. File Number: FA-2025-00847."),
    ("What does homeowners insurance cost?", "Homeowners Insurance Premium: $1,500.00 annually ($125.00/month through escrow)."),
    ("Am I required to carry mortgage insurance?", "Mortgage Insurance: $0.00 per month. Private mortgage insurance is not required with 20% down."),
    ("What is the finance charge on the TILA?", "Finance Charge: $129,059.36. This represents the total dollar cost of credit over the loan term."),
    ("What is the amount financed?", "Amount Financed: $186,147.50. This is the loan proceeds provided after subtracting prepaid charges."),
    ("Show me the payment schedule", "Payment Schedule: 119 payments of $1,049.11 monthly from 12/01/05 through 11/01/15."),
    ("What are the total settlement charges on the GFE?", "Total Estimated Settlement Charges: $7,658.43 as shown on the Good Faith Estimate."),
    ("What credit score does the borrower have?", "Credit score information is not included in closing disclosures or promissory notes."),
    ("What is the borrower's annual income?", "Borrower income is not disclosed on these documents. Income verification is separate."),
    ("Do the numbers match between the estimate and disclosure?", "The Loan Estimate and Closing Disclosure show consistent figures: Loan Amount $240,000."),
    ("What is the down payment percentage?", "Down Payment: $60,000.00 representing 20% of the $300,000 sale price."),
    ("What are the prepaid items at closing?", "Prepaids totaling $2,000.00: Homeowners Insurance (12 months) $1,500.00, Interest $500.00."),
    ("What is the LTV ratio?", "Loan-to-Value ratio: 80%. Calculated as $240,000 loan divided by $300,000 property value."),
    ("What title insurance costs are there?", "Title charges: Lenders Title Insurance $1,025.00, Settlement Fee $750.00, Search Fee $400.00."),
    ("What are the monthly escrow requirements?", "Monthly escrow deposit of $387.50 covers property tax at $262.50 and hazard insurance at $125.00."),
    ("What type of property is being financed?", "Property Type: Single Family Residence. Occupancy: Primary Residence. Built: 1998."),
    ("Is flood insurance required?", "Flood determination indicates the property is not in a special flood hazard area. No flood insurance required."),
    ("What is the loan-to-value ratio after closing?", "Post-closing LTV: 80.00%. Combined LTV: 80.00%. No subordinate financing."),
]

print(f"{len(MORTGAGE_PAIRS)} training pairs prepared")
train_samples = [InputExample(texts=[q, p]) for q, p in MORTGAGE_PAIRS]
train_loader = DataLoader(train_samples, shuffle=True, batch_size=16)

ft_encoder = SentenceTransformer(CFG["embed_model"], device="cuda")
loss_fn = losses.MultipleNegativesRankingLoss(ft_encoder)

N_EPOCHS = 10
warmup = int(len(train_loader) * N_EPOCHS * 0.1)
FT_MODEL_PATH = "/content/models/finance-tuned-embeddings"

print(f"Fine-tuning: {N_EPOCHS} epochs, {len(train_loader)} batches/epoch")
ft_encoder.fit(
    train_objectives=[(train_loader, loss_fn)],
    epochs=N_EPOCHS,
    warmup_steps=warmup,
    output_path=FT_MODEL_PATH,
    show_progress_bar=True,
)
print(f"Saved fine-tuned model: {FT_MODEL_PATH}")

tuned_encoder = SentenceTransformer(FT_MODEL_PATH, device="cuda")
test_qs = ["What is the interest rate?", "How much are closing costs?", "Is there a prepayment penalty?", "What is the monthly escrow?"]
test_ps = [p for _, p in MORTGAGE_PAIRS[:12]]

base_qv = encoder.encode(test_qs, convert_to_numpy=True)
base_pv = encoder.encode(test_ps, convert_to_numpy=True)
tuned_qv = tuned_encoder.encode(test_qs, convert_to_numpy=True)
tuned_pv = tuned_encoder.encode(test_ps, convert_to_numpy=True)

print(f"\n{'Query':<40} {'Base Top':<12} {'Tuned Top':<12}")
print("=" * 64)
for i, q in enumerate(test_qs):
    b = int(np.argmax(base_qv[i] @ base_pv.T))
    t = int(np.argmax(tuned_qv[i] @ tuned_pv.T))
    print(f"{q:<40} Passage {b:<5} Passage {t:<5}")

encoder = tuned_encoder
print("\nGlobal encoder replaced with fine-tuned version.")

35 training pairs prepared


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fine-tuning: 10 epochs, 3 batches/epoch


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned model: /content/models/finance-tuned-embeddings


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Query                                    Base Top     Tuned Top   
What is the interest rate?               Passage 1     Passage 0    
How much are closing costs?              Passage 6     Passage 6    
Is there a prepayment penalty?           Passage 8     Passage 8    
What is the monthly escrow?              Passage 4     Passage 4    

Global encoder replaced with fine-tuned version.


In [ ]:
# CELL 5: DATA MODELS
# define PageData for each PDF page
# define DocSection for grouped pages of same document type
# define Chunk for retrieval-ready text pieces
# keep pipeline data organized and easy to track
@dataclass
class PageData:
    idx: int
    text: str
    label: Optional[str] = None

@dataclass
class DocSection:
    section_id: str
    label: str
    start_page: int
    end_page: int
    raw_text: str

@dataclass
class Chunk:
    chunk_id: str
    section_id: str
    label: str
    seq: int
    page_start: int
    page_end: int
    text: str
    embedding: Optional[np.ndarray] = None

print("Data models: PageData, DocSection, Chunk")

Data models: PageData, DocSection, Chunk


In [ ]:
# CELL 6: ZERO-SHOT EMBEDDING CLASSIFIER
# define anchor descriptions for each document type
# create average embedding for each category
# classify each page by similarity to category anchors
# return "Other" if confidence is too low
# check page-number continuity to keep multi-page docs together
# decide whether adjacent pages belong to same section

CATEGORY_ANCHORS = {
    "Mortgage Note": [
        "promissory note with fixed interest rate and monthly payments",
        "deed of trust securing a residential mortgage loan",
        "borrower agrees to repay principal and interest over 30 years",
    ],
    "Closing Disclosure": [
        "closing disclosure form showing final loan terms and costs",
        "cash to close breakdown with projected monthly payments",
        "closing cost details including lender and third-party fees",
    ],
    "Good Faith Estimate": [
        "good faith estimate of settlement charges from lender",
        "origination fees and estimated closing costs worksheet",
        "HUD settlement statement itemizing buyer and seller charges",
    ],
    "Pay Stub": [
        "employee pay stub showing gross pay net pay and deductions",
        "payroll earnings statement with withholding and YTD totals",
    ],
    "Bank Statement": [
        "monthly bank account statement with deposits and withdrawals",
        "checking or savings account ending balance and transactions",
    ],
    "Tax Document": [
        "IRS form W-2 showing wages and tax withholding",
        "federal income tax return form 1040 with filing status",
    ],
    "Insurance Policy": [
        "homeowners insurance policy with coverage limits and premium",
        "property insurance declaration page with deductible amounts",
    ],
    "Deed": [
        "warranty deed transferring real property from grantor to grantee",
        "recorded deed with legal description and parcel number",
    ],
    "Invoice": [
        "invoice with line items quantities unit prices and total due",
        "billing statement with payment terms and balance due",
    ],
    "Contract": [
        "legal agreement between parties with terms and obligations",
        "contract with governing law termination and confidentiality clauses",
    ],
    "Letter": [
        "formal letter with salutation body and closing signature",
        "correspondence addressed to a specific recipient",
    ],
    "Report": [
        "analytical report with executive summary methodology and findings",
        "research document with conclusions and recommendations",
    ],
}
_cat_labels = list(CATEGORY_ANCHORS.keys())
_cat_ref_vecs = []
for cat in _cat_labels:
    vecs = encoder.encode(CATEGORY_ANCHORS[cat], convert_to_numpy=True)
    _cat_ref_vecs.append(vecs.mean(axis=0))
_cat_ref_matrix = np.stack(_cat_ref_vecs)
_cat_ref_norms = _cat_ref_matrix / (np.linalg.norm(_cat_ref_matrix, axis=1, keepdims=True) + 1e-9)

def classify_page(text: str, min_confidence: float = 0.25) -> str:
    if not text or len(text.strip()) < 20:
        return "Other"
    page_vec = encoder.encode([text[:1500]], convert_to_numpy=True)[0]
    page_norm = page_vec / (np.linalg.norm(page_vec) + 1e-9)
    similarities = _cat_ref_norms @ page_norm
    best_idx = int(np.argmax(similarities))
    best_score = float(similarities[best_idx])
    if best_score < min_confidence:
        return "Other"
    return _cat_labels[best_idx]

_PG_RE = re.compile(r"\bpage\s+(\d+)\s+of\s+(\d+)\b", re.IGNORECASE)
def _pages_continuous(prev: str, curr: str) -> bool:
    p = _PG_RE.search((prev or "")[-300:])
    c = _PG_RE.search((curr or "")[:300])
    if not (p and c):
        return False
    try:
        return int(c.group(1)) == int(p.group(1)) + 1 and c.group(2) == p.group(2)
    except (ValueError, IndexError):
        return False

def same_section(prev_text: str, curr_text: str, running_label: str) -> bool:
    if _pages_continuous(prev_text, curr_text):
        return True
    return classify_page(curr_text) == running_label

print(f"Zero-shot classifier ready. {len(_cat_labels)} categories: {_cat_labels}")

Zero-shot classifier ready. 12 categories: ['Mortgage Note', 'Closing Disclosure', 'Good Faith Estimate', 'Pay Stub', 'Bank Statement', 'Tax Document', 'Insurance Policy', 'Deed', 'Invoice', 'Contract', 'Letter', 'Report']


In [ ]:
# CELL 7: PDF EXTRACTION + OCR FALLBACK + SEGMENTATION
# open PDF from file path or uploaded content
# extract text from each page directly
# if page has too little text, run OCR on page image
# store each page as PageData
# classify first page
# group consecutive pages into sections
# split sections when document type changes
OCR_THRESHOLD = 30

def extract_pdf(pdf_input) -> Tuple[List[PageData], List[DocSection]]:
    if isinstance(pdf_input, dict) and "content" in pdf_input:
        doc = fitz.open(stream=pdf_input["content"], filetype="pdf")
    elif hasattr(pdf_input, "read"):
        doc = fitz.open(stream=pdf_input.read(), filetype="pdf")
    else:
        doc = fitz.open(pdf_input)

    pages = []
    for i in range(len(doc)):
        pg = doc[i]
        text = pg.get_text() or ""
        if len(text.strip()) < OCR_THRESHOLD:
            try:
                pix = pg.get_pixmap(dpi=200)
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(img) or ""
            except Exception:
                text = ""
        pages.append(PageData(idx=i, text=text))
    doc.close()

    if not pages:
        raise ValueError("No extractable content in PDF")

    sections = []
    buf = []
    cur_label = None
    sec_ct = 0

    for i, pg in enumerate(pages):
        if i == 0:
            cur_label = classify_page(pg.text)
            pg.label = cur_label
            buf = [pg]
        else:
            if same_section(pages[i-1].text, pg.text, cur_label):
                pg.label = cur_label
                buf.append(pg)
            else:
                sections.append(DocSection(
                    section_id=f"sec_{sec_ct}",
                    label=cur_label,
                    start_page=buf[0].idx,
                    end_page=buf[-1].idx,
                    raw_text="\n\n".join(p.text for p in buf),
                ))
                sec_ct += 1
                cur_label = classify_page(pg.text)
                pg.label = cur_label
                buf = [pg]

    if buf:
        sections.append(DocSection(
            section_id=f"sec_{sec_ct}",
            label=cur_label,
            start_page=buf[0].idx,
            end_page=buf[-1].idx,
            raw_text="\n\n".join(p.text for p in buf),
        ))

    return pages, sections

print("PDF extractor ready (digital + OCR + embedding-based segmentation)")

PDF extractor ready (digital + OCR + embedding-based segmentation)


In [ ]:
# CELL 8: SEMANTIC CHUNKING
# normalize embedding vectors
# split section text into sentences
# embed sentences and compare neighboring sentence similarity
# create breakpoints when topic changes
# build chunks with min / max sentence and character limits
# estimate page range for each chunk
# chunk all sections for retrieval
def _normalize_vectors(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9
    return vectors / norms

def _semantic_breakpoints(sentences: List[str], similarity_threshold: float = 0.72) -> List[int]:
    if len(sentences) <= 1:
        return []
    sent_vecs = encoder.encode(sentences, convert_to_numpy=True)
    sent_vecs = _normalize_vectors(sent_vecs)
    breakpoints = []
    for i in range(len(sentences) - 1):
        sim = float(np.dot(sent_vecs[i], sent_vecs[i + 1]))
        if sim < similarity_threshold:
            breakpoints.append(i + 1)
    return breakpoints

def chunk_section(section: DocSection,
                  similarity_threshold: float = None,
                  min_sentences: int = None,
                  max_sentences: int = None,
                  min_chars: int = None,
                  max_chars: int = None) -> List[Chunk]:
    similarity_threshold = similarity_threshold or CFG["semantic_similarity_threshold"]
    min_sentences = min_sentences or CFG["chunk_min_sentences"]
    max_sentences = max_sentences or CFG["chunk_max_sentences"]
    min_chars = min_chars or CFG["chunk_min_chars"]
    max_chars = max_chars or CFG["chunk_max_chars"]

    sentences = sent_tokenize(section.raw_text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

    if not sentences:
        return [Chunk(
            chunk_id=f"{section.section_id}_c0",
            section_id=section.section_id,
            label=section.label,
            seq=0,
            page_start=section.start_page,
            page_end=section.end_page,
            text=section.raw_text,
        )]

    if len(sentences) <= min_sentences:
        return [Chunk(
            chunk_id=f"{section.section_id}_c0",
            section_id=section.section_id,
            label=section.label,
            seq=0,
            page_start=section.start_page,
            page_end=section.end_page,
            text=" ".join(sentences),
        )]

    breakpoints = set(_semantic_breakpoints(sentences, similarity_threshold))
    chunks = []
    current = []
    current_char_count = 0
    total_chars = sum(len(s) for s in sentences)

    char_offsets = []
    running = 0
    for s in sentences:
        char_offsets.append(running)
        running += len(s)

    chunk_start_idx = 0

    for i, sent in enumerate(sentences):
        current.append(sent)
        current_char_count += len(sent)

        current_len = len(current)
        next_is_break = (i + 1) in breakpoints
        hit_max_sentences = current_len >= max_sentences
        hit_max_chars = current_char_count >= max_chars
        enough_size = (current_len >= min_sentences and current_char_count >= min_chars)

        should_close = False
        if hit_max_sentences or hit_max_chars:
            should_close = True
        elif next_is_break and enough_size:
            should_close = True

        if should_close:
            chunk_text = " ".join(current)
            progress = char_offsets[chunk_start_idx] / max(1, total_chars)
            span = max(1, section.end_page - section.start_page + 1)
            est_start = section.start_page + int(progress * span)
            est_end = min(section.end_page, est_start + 1)

            chunks.append(Chunk(
                chunk_id=f"{section.section_id}_c{len(chunks)}",
                section_id=section.section_id,
                label=section.label,
                seq=len(chunks),
                page_start=est_start,
                page_end=est_end,
                text=chunk_text,
            ))

            current = []
            current_char_count = 0
            chunk_start_idx = i + 1

    if current:
        chunk_text = " ".join(current)
        progress = char_offsets[chunk_start_idx] / max(1, total_chars) if chunk_start_idx < len(char_offsets) else 0
        span = max(1, section.end_page - section.start_page + 1)
        est_start = section.start_page + int(progress * span)
        est_end = min(section.end_page, est_start + 1)

        chunks.append(Chunk(
            chunk_id=f"{section.section_id}_c{len(chunks)}",
            section_id=section.section_id,
            label=section.label,
            seq=len(chunks),
            page_start=est_start,
            page_end=est_end,
            text=chunk_text,
        ))

    return chunks

def chunk_all(sections: List[DocSection]) -> List[Chunk]:
    all_chunks = []
    for sec in sections:
        all_chunks.extend(chunk_section(sec))
    return all_chunks

print("Semantic chunker ready.")

Semantic chunker ready.


In [ ]:
# CELL 9: ENTITY EXTRACTION + NORMALIZATION
# define helper functions to normalize currency, rates, dates, and names
# define which fields should be normalized
# build prompt asking LLM to return JSON only
# extract structured finance entities from text
# parse LLM JSON output
# normalize extracted values into clean machine-friendly format
# merge entity results across sections
# render extracted entities as a markdown table

def _norm_currency(raw: str) -> Optional[float]:
    if not raw: return None
    digits = re.sub(r"[^\d.]", "", raw.replace(",", ""))
    try: return round(float(digits), 2)
    except ValueError: return None

def _norm_rate(raw: str) -> Optional[float]:
    if not raw: return None
    cleaned = raw.lower().replace("percent", "").replace("%", "").strip()
    try:
        v = float(cleaned)
        return round(v / 100, 6) if v > 1 else round(v, 6)
    except ValueError:
        return None

def _norm_date(raw: str) -> Optional[str]:
    if not raw: return None
    for fmt in ["%m/%d/%Y", "%m/%d/%y", "%B %d, %Y", "%b %d, %Y", "%Y-%m-%d", "%m-%d-%Y", "%d %B %Y", "%B %Y"]:
        try:
            return datetime.strptime(raw.strip(), fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return raw.strip()

def _norm_name(raw: str) -> Optional[str]:
    return raw.strip().title() if raw else None

NORMALIZERS = {
    "loan_amount": _norm_currency, "monthly_payment": _norm_currency,
    "closing_costs": _norm_currency, "down_payment": _norm_currency,
    "cash_to_close": _norm_currency, "finance_charge": _norm_currency,
    "total_payments": _norm_currency, "escrow_monthly": _norm_currency,
    "interest_rate": _norm_rate, "apr": _norm_rate,
    "closing_date": _norm_date, "first_payment_date": _norm_date,
    "maturity_date": _norm_date, "rate_lock_expiration": _norm_date,
    "borrower_name": _norm_name, "lender_name": _norm_name,
    "settlement_agent": _norm_name,
}

EXTRACTION_PROMPT = """Extract financial entities from this mortgage document text.
Return ONLY a JSON object with null for missing fields. No explanation.

Fields: borrower_name, lender_name, property_address, loan_amount, interest_rate,
monthly_payment, loan_term, loan_type, closing_date, first_payment_date,
maturity_date, apr, total_payments, closing_costs, down_payment,
cash_to_close, prepayment_penalty, late_charge, escrow_monthly,
finance_charge, loan_number, settlement_agent

TEXT:
{text}

JSON:"""

def extract_entities(text: str, limit: int = 3000) -> Dict:
    prompt = EXTRACTION_PROMPT.format(text=text[:limit])
    try:
        out = generate(prompt, max_tokens=600, temp=0.05)
        out = re.sub(r"^```json\s*", "", out.strip())
        out = re.sub(r"\s*```$", "", out)
        m = re.search(r"\{[\s\S]*\}", out)
        if not m:
            return {"_error": "No JSON found in output"}
        raw = json.loads(m.group())
        result = {}
        for k, v in raw.items():
            if v is not None:
                fn = NORMALIZERS.get(k)
                norm = fn(str(v)) if fn else str(v)
                result[k] = {"raw": str(v), "normalized": norm}
            else:
                result[k] = {"raw": None, "normalized": None}
        return result
    except json.JSONDecodeError as e:
        return {"_error": f"JSON parse: {e}"}
    except Exception as e:
        return {"_error": str(e)}

def extract_all_sections(sections: List[DocSection]) -> Dict:
    merged = {}
    for sec in sections:
        print(f"  Extracting: {sec.section_id} ({sec.label})...")
        ents = extract_entities(sec.raw_text)
        if "_error" in ents:
            print(f"    Warn: {ents['_error']}")
            continue
        for k, v in ents.items():
            if k.startswith("_"): continue
            if k not in merged or merged[k].get("normalized") is None:
                merged[k] = v
    return merged

def render_entity_table(entities: Dict) -> str:
    if not entities or "_error" in entities:
        return "No entities extracted."
    rows = ["| Field | Value | Normalized |", "|---|---|---|"]
    for k in sorted(entities):
        if k.startswith("_"): continue
        v = entities[k]
        if isinstance(v, dict):
            raw = v.get("raw") or "—"
            norm = v.get("normalized")
            rows.append(f"| {k.replace('_',' ').title()} | {raw} | {norm if norm else '—'} |")
    return "\n".join(rows)

print("Entity extraction + normalization ready.")

Entity extraction + normalization ready.


In [ ]:
# CELL 10: HYBRID RETRIEVER (SEMANTIC + KEYWORD SEARCH)
# define finance-specific keyword boost terms
# tokenize query and chunk text for keyword scoring
# compute keyword overlap score
# boost chunks that contain useful finance patterns like percentages and dollar amounts
# build FAISS semantic index over chunk embeddings
# optionally build per-document-type sub-indexes
# retrieve candidates from semantic search
# retrieve candidates from keyword search
# merge both scores into one hybrid ranking
from collections import Counter
import re
from typing import List

FINANCE_BOOST_TERMS = {
    "interest": 0.20,
    "rate": 0.20,
    "apr": 0.25,
    "principal": 0.18,
    "escrow": 0.18,
    "borrower": 0.15,
    "lender": 0.15,
    "parcel": 0.15,
    "mortgage": 0.12,
    "note": 0.12,
    "loan": 0.12,
    "payment": 0.12,
    "late": 0.10,
    "charge": 0.10,
}

def _tokenize_for_keyword_search(text: str) -> List[str]:
    text = text.lower()
    tokens = re.findall(r"[a-zA-Z0-9%$.\-]+", text)
    return [t for t in tokens if len(t) > 1]

def _keyword_overlap_score(query: str, text: str) -> float:
    q_tokens = _tokenize_for_keyword_search(query)
    t_tokens = _tokenize_for_keyword_search(text)

    if not q_tokens or not t_tokens:
        return 0.0

    q_counts = Counter(q_tokens)
    t_counts = Counter(t_tokens)

    overlap = 0.0
    for tok, q_count in q_counts.items():
        overlap += min(q_count, t_counts.get(tok, 0))

    base_score = overlap / max(1, len(set(q_tokens)))

    boost = 0.0
    q_token_set = set(q_tokens)
    t_token_set = set(t_tokens)

    for term, weight in FINANCE_BOOST_TERMS.items():
        if term in q_token_set and term in t_token_set:
            boost += weight

    # Bonus for percentages and dollar amounts in finance answers
    if re.search(r"\d+(\.\d+)?\s*%", text):
        boost += 0.15
    if re.search(r"\$\s?\d[\d,]*(\.\d+)?", text):
        boost += 0.10

    return base_score + boost

class HybridRetriever:
    def __init__(self):
        self.global_idx = None
        self.chunks: List[Chunk] = []
        self.section_indices: Dict[str, Dict] = {}
        self._normed_vecs = None

    def index(self, chunks: List[Chunk]):
        self.chunks = chunks
        texts = [c.text for c in chunks]
        raw_vecs = encoder.encode(texts, show_progress_bar=True, convert_to_numpy=True)
        normed = _normalize_vectors(raw_vecs).astype(np.float32)
        self._normed_vecs = normed

        for i, c in enumerate(chunks):
            c.embedding = normed[i]

        dim = normed.shape[1]
        self.global_idx = faiss.IndexFlatIP(dim)
        self.global_idx.add(normed)

        self.section_indices = {}
        for lbl in sorted(set(c.label for c in chunks)):
            member_ids = [i for i, c in enumerate(chunks) if c.label == lbl]
            if not member_ids:
                continue
            sub = faiss.IndexFlatIP(dim)
            sub.add(normed[member_ids])
            self.section_indices[lbl] = {"index": sub, "map": member_ids}
        print(f"Indexed {len(chunks)} chunks with hybrid retrieval")

    def _semantic_candidates(self, query: str, k: int, label_filter: Optional[str] = None):
        q_vec = encoder.encode([query], convert_to_numpy=True)
        q_norm = _normalize_vectors(q_vec).astype(np.float32)
        if label_filter and label_filter in self.section_indices:
            si = self.section_indices[label_filter]
            scores, local_ids = si["index"].search(q_norm, k)
            ids = [si["map"][li] for li in local_ids[0]]
            scs = scores[0]
        else:
            scores, ids_arr = self.global_idx.search(q_norm, k)
            ids = ids_arr[0]
            scs = scores[0]
        return [(gid, float(scs[j])) for j, gid in enumerate(ids) if gid >= 0]

    def _keyword_candidates(self, query: str, k: int, label_filter: Optional[str] = None):
        scored = []
        for i, chunk in enumerate(self.chunks):
            if label_filter and chunk.label != label_filter:
                continue
            kw_score = _keyword_overlap_score(query, chunk.text)
            if kw_score > 0:
                scored.append((i, kw_score))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:k]

    def search(self, query: str, top_k: int = None,
               label_filter: Optional[str] = None,
               alpha: float = None,
               candidate_pool: int = None) -> List[Tuple[Chunk, float]]:
        k = top_k or CFG["retrieval_k"]
        alpha = alpha if alpha is not None else CFG["hybrid_alpha"]
        candidate_pool = candidate_pool or CFG["hybrid_candidate_pool"]

        semantic_hits = self._semantic_candidates(query, candidate_pool, label_filter)
        keyword_hits = self._keyword_candidates(query, candidate_pool, label_filter)

        semantic_map = {idx: score for idx, score in semantic_hits}
        keyword_map = {idx: score for idx, score in keyword_hits}
        all_ids = set(semantic_map.keys()) | set(keyword_map.keys())

        merged = []
        for idx in all_ids:
            sem_score = semantic_map.get(idx, 0.0)
            kw_score = keyword_map.get(idx, 0.0)
            hybrid_score = alpha * sem_score + (1 - alpha) * kw_score
            merged.append((self.chunks[idx], hybrid_score))

        merged.sort(key=lambda x: x[1], reverse=True)
        return merged[:k]

print("HybridRetriever ready (semantic + keyword search)")

HybridRetriever ready (semantic + keyword search)


In [ ]:
# CELL 11: GROUNDED ANSWER GENERATION
# estimate token size of context blocks
# format page references nicely
# fit retrieved context into token budget
# remove duplicate chunks
# build answer context from top retrieved chunks
# create prompt telling LLM to answer only from provided context
# return answer, sources, and confidence
# also support full-document summarization
def _token_est(text: str) -> int:
    return max(1, len(text) // 4)

def format_page_range(page_start: int, page_end: int) -> str:
    start = page_start + 1
    end = page_end + 1
    if start == end:
        return f"Page {start}"
    return f"Pages {start}-{end}"

def _budget_fit(blocks: List[str], limit: int) -> str:
    out, used = [], 0
    for b in blocks:
        t = _token_est(b)
        if used + t > limit:
            remaining = limit - used
            out.append(b[:remaining * 4])
            break
        out.append(b)
        used += t
    return "\n\n".join(out)

def answer_question(question: str, hits: List[Tuple[Chunk, float]]) -> Dict:
    if not hits:
        return {"answer": "No relevant content found.", "sources": [], "confidence": 0.0}

    seen, unique = set(), []
    for chunk, score in hits:
        if chunk.chunk_id not in seen:
            seen.add(chunk.chunk_id)
            unique.append((chunk, score))

    ctx_blocks, sources = [], []
    for chunk, score in unique:
        page_label = format_page_range(chunk.page_start, chunk.page_end)
        header = f"[{chunk.label} | {page_label} | sim={score:.2f}]"
        ctx_blocks.append(f"{header}\n{chunk.text}")
        sources.append({
            "doc_type": chunk.label,
            "pages": format_page_range(chunk.page_start, chunk.page_end),
            "relevance": f"{score:.0%}",
            "preview": chunk.text[:150] + "..." if len(chunk.text) > 150 else chunk.text,
        })

    budget = max(256, CFG["ctx_window"] - CFG["max_gen_tokens"] - 256)
    context = _budget_fit(ctx_blocks, budget)

    prompt = f"""[INST]
You are a financial document assistant.

Rules:
1. Answer only from the provided context.
2. Be concise and factual.
3. If the exact answer is not stated, say the information is not clearly stated in the provided pages.
4. Prefer exact values, dates, names, and amounts when present.
5. End with a short page reference like "(Pages 2-3)".

Context:
{context}

Question: {question}
[/INST]
Answer:"""

    try:
        text = generate(prompt)
        avg = sum(s for _, s in unique) / max(1, len(unique))
        return {"answer": text, "sources": sources, "confidence": avg}
    except Exception as e:
        return {"answer": f"Generation error: {e}", "sources": sources, "confidence": 0.0}

def summarize_sections(sections: List[DocSection]) -> Dict:
    if not sections:
        return {"answer": "No documents loaded.", "sources": [], "confidence": None}
    blocks = []
    for sec in sections:
        hdr = f"[{sec.label} | Pages {sec.start_page+1}-{sec.end_page+1}]"
        preview = sec.raw_text[:1200] + "..." if len(sec.raw_text) > 1200 else sec.raw_text
        blocks.append(f"{hdr}\n{preview}")

    budget = max(256, CFG["ctx_window"] - 512)
    context = _budget_fit(blocks, budget)

    prompt = f"""Summarize this multi-document financial packet for a non-expert.
Organize by document type. Be concise and reference key amounts with page numbers.

Documents:
{context}

Summary:"""

    text = generate(prompt, max_tokens=600)
    sources = [{"doc_type": s.label, "pages": f"{s.start_page+1}-{s.end_page+1}", "relevance": "", "preview": s.raw_text[:150] + "..."} for s in sections]
    return {"answer": text.strip(), "sources": sources, "confidence": None}

print("Answer generation ready.")

Answer generation ready.


In [ ]:
# CELL 12: PIPELINE ORCHESTRATOR
# define phrase hints for likely document types
# guess best doc type for a user query
# create FinanceRAG class to manage full workflow
# ingest PDF
# extract pages and sections
# chunk sections
# build retrieval index
# extract entities
# store metadata like page count, chunk count, doc types, and timing
# track whether pipeline is ready for questions

DOC_TYPE_HINTS = {
    "Closing Disclosure": [
        "closing cost", "closing costs", "cash to close", "apr",
        "projected payments", "closing disclosure"
    ],
    "Good Faith Estimate": [
        "settlement charges", "good faith estimate", "gfe",
        "origination fees", "title charges"
    ],
    "Mortgage Note": [
        "interest rate", "late charge", "prepayment penalty",
        "loan term", "monthly payment", "note"
    ],
    "Pay Stub": [
        "gross pay", "net pay", "earnings", "deductions", "pay stub"
    ],
    "Bank Statement": [
        "ending balance", "deposits", "withdrawals", "bank statement"
    ],
    "Tax Document": [
        "1098", "1099", "w-2", "1040", "irs", "tax"
    ],
    "Insurance Policy": [
        "premium", "coverage", "deductible", "insurance"
    ],
    "Deed": [
        "grantor", "grantee", "parcel", "legal description", "deed"
    ],
}

def suggest_doc_type_for_query(question: str, available_types: List[str]) -> Optional[str]:
    q = question.lower().strip()
    best_type = None
    best_score = 0

    for doc_type, phrases in DOC_TYPE_HINTS.items():
        if doc_type not in available_types:
            continue
        score = 0
        for phrase in phrases:
            if phrase in q:
                score += len(phrase.split())

        if score > best_score:
            best_score = score
            best_type = doc_type

    return best_type if best_score > 0 else None

class FinanceRAG:
    def __init__(self):
        self.pages: List[PageData] = []
        self.sections: List[DocSection] = []
        self.chunks: List[Chunk] = []
        self.retriever = HybridRetriever()
        self.entities: Dict = {}
        self.ready = False
        self.meta: Dict = {}
        self.filename: Optional[str] = None

    def ingest(self, pdf_input, filename: str = "document.pdf") -> Tuple[bool, Dict]:
        self.ready = False
        self.filename = filename
        t0 = datetime.now()

        self.pages, self.sections = extract_pdf(pdf_input)
        self.chunks = chunk_all(self.sections)
        self.retriever.index(self.chunks)

        print("Extracting entities...")
        try:
            self.entities = extract_all_sections(self.sections)
            n_ents = sum(1 for v in self.entities.values() if isinstance(v, dict) and v.get("normalized") is not None)
            print(f"  {n_ents} entities normalized")
        except Exception as e:
            print(f"  Entity extraction warning: {e}")
            self.entities = {}

        elapsed = (datetime.now() - t0).total_seconds()
        labels = sorted(set(s.label for s in self.sections))
        self.meta = {
            "filename": filename,
            "pages": len(self.pages),
            "sections": len(self.sections),
            "chunks": len(self.chunks),
            "doc_types": labels,
            "entities_found": sum(1 for v in self.entities.values() if isinstance(v, dict) and v.get("normalized") is not None),
            "time": f"{elapsed:.1f}s",
        }
        self.ready = True
        return True, self.meta

    def ask(self, question: str, top_k: int = 5, label_filter: Optional[str] = None) -> Dict:
        if not self.ready:
            return {"answer": "Upload a PDF first.", "sources": [], "confidence": 0.0}

        active_filter = label_filter
        if active_filter is None:
            available_types = sorted(set(s.label for s in self.sections))
            active_filter = suggest_doc_type_for_query(question, available_types)

        hits = self.retriever.search(question, top_k=top_k, label_filter=active_filter)
        result = answer_question(question, hits)
        result["applied_filter"] = active_filter
        return result

    def summarize(self) -> Dict:
        if not self.ready:
            return {"answer": "No documents loaded.", "sources": [], "confidence": None}
        return summarize_sections(self.sections)

    def structure(self) -> List[Dict]:
        return [{"id": s.section_id, "type": s.label, "pages": f"{s.start_page+1}-{s.end_page+1}", "chunks": sum(1 for c in self.chunks if c.section_id == s.section_id)} for s in self.sections]

    def entity_table(self) -> str:
        return render_entity_table(self.entities)

pipeline = FinanceRAG()
print("FinanceRAG orchestrator ready.")

FinanceRAG orchestrator ready.


In [ ]:
# CELL 13: GRADIO UI + EVENT HANDLERS
# define suggested questions by document type
# generate a short list of suggested questions for the loaded PDF
# format sources for chat display
# define process button behavior
# define ask-question behavior
# define summarize button behavior
# define entity extraction display behavior
# define export-to-JSON behavior
# define suggested-question click behavior

TYPE_QUESTIONS = {
    "Mortgage Note": [
        "What is the interest rate?",
        "Is there a prepayment penalty?",
        "What is the late charge policy?",
    ],
    "Good Faith Estimate": [
        "What are the total settlement charges?",
        "List the origination fees.",
        "What are the title charges?",
    ],
    "Closing Disclosure": [
        "What are the total closing costs?",
        "What is the cash to close?",
        "What is the APR?",
    ],
    "Pay Stub": [
        "What is the net pay?",
        "List earnings and deductions.",
    ],
    "Invoice": [
        "What is the total amount due?",
    ],
}

def _suggestions(types):
    qs = []
    for t in types:
        qs.extend(TYPE_QUESTIONS.get(t, []))
    qs.append("What pages mention payment amounts?")
    seen = set()
    out = []
    for q in qs:
        if q not in seen:
            seen.add(q)
            out.append(q)
    return out[:8]

def _fmt_sources_simple(sources):
    if not sources:
        return ""

    seen = set()
    cleaned = []

    for s in sources:
        doc_type = s.get("doc_type", "Document")
        pages = s.get("pages", "?")
        key = (doc_type, pages)

        if key not in seen:
            seen.add(key)
            cleaned.append((doc_type, pages))

    lines = ["\n\nSources:"]
    for doc_type, pages in cleaned[:5]:
        lines.append(f"- {doc_type} | {pages}")

    return "\n".join(lines)

def on_process(pdf_file):
    if pdf_file is None:
        return (
            "Upload a PDF first.",
            "",
            gr.update(choices=["All"], value="All"),
            gr.update(choices=[], value=None),
            []
        )

    try:
        fname = pdf_file if isinstance(pdf_file, str) else getattr(pdf_file, "name", "doc.pdf")
        ok, meta = pipeline.ingest(fname, filename=os.path.basename(str(fname)))

        if ok:
            status = (
                f"Loaded: {meta['filename']}\n"
                f"Pages: {meta['pages']} | Sections: {meta['sections']} | Chunks: {meta['chunks']}\n"
                f"Document types: {', '.join(meta['doc_types'])}"
            )

            struct = "### Document Structure\n" + "\n".join(
                f"- **{d['type']}** | Pages {d['pages']}"
                for d in pipeline.structure()
            )

            filters = ["All"] + meta["doc_types"]
            suggs = _suggestions(meta["doc_types"])

            return (
                status,
                struct,
                gr.update(choices=filters, value="All"),
                gr.update(choices=suggs, value=suggs[0] if suggs else None),
                []
            )

        return (
            "Processing failed.",
            "",
            gr.update(choices=["All"], value="All"),
            gr.update(choices=[], value=None),
            []
        )

    except Exception as e:
        return (
            f"Error: {e}",
            "",
            gr.update(choices=["All"], value="All"),
            gr.update(choices=[], value=None),
            []
        )

def on_ask(message, history, doc_filter, num_chunks):
    history = history or []
    if not message or not message.strip():
        return history, ""
    if not pipeline.ready:
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": "Please upload and process a PDF first."})
        return history, ""
    filt = None if doc_filter == "All" else doc_filter
    try:
        res = pipeline.ask(message, top_k=num_chunks, label_filter=filt)
        reply = res.get("answer", "").strip()
        reply += _fmt_sources_simple(res.get("sources", []))
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": reply})
        return history, ""
    except Exception as e:
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": f"Error: {e}"})
        return history, ""

def on_summary(history):
    history = history or []
    if not pipeline.ready:
        history.append({"role": "user", "content": "Summarize"})
        history.append({"role": "assistant", "content": "Upload a PDF first."})
        return history
    res = pipeline.summarize()
    text = res.get("answer", "").strip()
    text += _fmt_sources_simple(res.get("sources", []))
    history.append({"role": "user", "content": "Summarize the documents"})
    history.append({"role": "assistant", "content": text})
    return history

def on_entities():
    if not pipeline.ready:
        return "Process a PDF first."
    return pipeline.entity_table()

def on_export():
    if not pipeline.entities:
        return None
    export = {}
    for k, v in pipeline.entities.items():
        if not k.startswith("_") and isinstance(v, dict):
            export[k] = {
                "raw": v.get("raw"),
                "normalized": v.get("normalized"),
            }
    path = "/content/extracted_entities.json"
    with open(path, "w") as f:
        json.dump(export, f, indent=2, default=str)
    return path

def on_ask_suggestion(q, history, doc_filter, num_chunks):
    if not q:
        return history or [], ""
    return on_ask(q, history, doc_filter, num_chunks)

print("Handlers ready.")

Handlers ready.


In [ ]:
# CELL 14: GRADIO APP LAYOUT (THREE PANELS)

# create Gradio Blocks app
# add title and short description
# add PDF upload input
# add process button
# add status display
# add chatbot area
# add question input
# add controls for summary, entities, export, and suggestions
# connect UI components to event handlers

with gr.Blocks(title="Finance Document RAG", fill_height=True) as app:
    gr.Markdown("# Financial PDF Q&A")
    gr.Markdown("Upload a PDF, process it, and ask questions with page-cited answers.")

    with gr.Row(equal_height=True):
        # LEFT PANEL: Upload + processing controls
        with gr.Column(scale=3):
            pdf_in = gr.File(
                label="Upload PDF",
                file_types=[".pdf"],
                type="filepath",
                height=100
            )

            proc_btn = gr.Button("Process PDF", variant="primary")

            status_box = gr.Textbox(
                label="Status",
                interactive=False,
                lines=4
            )

            doc_filter = gr.Dropdown(
                choices=["All"],
                value="All",
                label="Document Type Filter"
            )

            num_chunks = gr.Slider(
                1, 8, value=4, step=1,
                label="Retrieved Chunks"
            )

        # MIDDLE PANEL: Chat
        with gr.Column(scale=5):
            chat = gr.Chatbot(
                height=280,
                label="Chat"
            )

            question = gr.Textbox(
                label="Ask a Question",
                placeholder="Example: What is the interest rate?"
            )

            with gr.Row():
                send_btn = gr.Button("Ask", variant="primary")
                clear_btn = gr.Button("Clear Chat")


        # RIGHT PANEL: Suggested questions + actions + outputs
        with gr.Column(scale=3):
            sugg_dd = gr.Dropdown(
                choices=[],
                value=None,
                label="Suggested Question"
            )

            ask_sugg_btn = gr.Button("Ask Suggested Question")

            with gr.Row():
                summ_btn = gr.Button("Summarize")
                ent_btn = gr.Button("Extract Entities")

            export_btn = gr.Button("Export JSON")
            export_out = gr.File(label="Downloaded JSON")

            struct_box = gr.Markdown("### Document Structure")
            ent_out = gr.Markdown("### Extracted Entities")

    # Wiring
    proc_btn.click(
        on_process,
        inputs=pdf_in,
        outputs=[status_box, struct_box, doc_filter, sugg_dd, chat]
    )

    send_btn.click(
        on_ask,
        inputs=[question, chat, doc_filter, num_chunks],
        outputs=[chat, question]
    )

    question.submit(
        on_ask,
        inputs=[question, chat, doc_filter, num_chunks],
        outputs=[chat, question]
    )

    ask_sugg_btn.click(
        on_ask_suggestion,
        inputs=[sugg_dd, chat, doc_filter, num_chunks],
        outputs=[chat, question]
    )

    summ_btn.click(
        on_summary,
        inputs=chat,
        outputs=chat
    )

    ent_btn.click(
        on_entities,
        outputs=ent_out
    )

    export_btn.click(
        on_export,
        outputs=export_out
    )

    clear_btn.click(
        lambda: [],
        outputs=chat
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bbcc2528fe3976e932.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
